In [10]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

In [3]:
df = pd.read_csv('processed_data.csv')
df.head()

,Target,Age,Gender,BMI,Systolic_BP,Diastolic_BP,Heart_Rate,Serum_Creatinine,eGFR,Urine_Albumin,...,Fasting_Glucose,HbA1c,Cholesterol,Triglycerides,Serum_Albumin,Total_Protein,Diabetes,Hypertension,Smoking_Status,Family_History_Kidney
0,Healthy Kidney,29,1,28,97,69,99,0,95,6,...,96,7.547874,204,120,4,7.091259,1,1,1,1
1,Severe CKD (Stage 4),43,0,18,165,100,67,5,28,318,...,88,7.287338,166,277,2,7.875167,1,1,1,0
2,Healthy Kidney,77,0,32,116,63,101,0,100,1,...,82,9.114854,246,299,4,7.083558,0,0,1,0
3,Healthy Kidney,83,0,24,93,75,87,0,101,11,...,82,7.286450,173,285,4,6.428780,1,0,0,1
4,Healthy Kidney,38,1,19,111,70,92,0,102,9,...,106,8.376492,266,294,4,7.852894,1,0,1,0


In [8]:
# Ordered CKD stage categories
stage_order = [
    "Healthy Kidney",
    "Mild CKD (Stage 1–2)",
    "Moderate CKD (Stage 3)",
    "Severe CKD (Stage 4)",
    "Kidney Failure (Stage 5)",
]
df["Target"] = pd.Categorical(df["Target"], categories=stage_order, ordered=True)
df_sorted = df.sort_values("Target")
 
# Consistent colour palette (one per stage)
STAGE_COLORS = {
    "Healthy Kidney":           "#2ecc71",
    "Mild CKD (Stage 1–2)":     "#f1c40f",
    "Moderate CKD (Stage 3)":   "#e67e22",
    "Severe CKD (Stage 4)":     "#e74c3c",
    "Kidney Failure (Stage 5)": "#8e44ad",
}

### 1 - Do Calcium and Phosphorus levels move in opposite directions as kidney disease worsens?

### 2 - How does this affect bone health?

In [7]:
df['Target'].unique()

array(['Healthy Kidney', 'Severe CKD (Stage 4)', 'Mild CKD (Stage 1–2)',
       'Moderate CKD (Stage 3)', 'Kidney Failure (Stage 5)'], dtype=object)

In [21]:
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Calcium & Phosphorus Levels by CKD Stage",
        "Bone Health Proxy: Ca × P Product by CKD Stage",
    ),
    horizontal_spacing=0.12,
)
 
stage_means = (
    df_sorted.groupby("Target", observed=True)[["Calcium", "Phosphorus"]]
    .mean()
    .reset_index()
)
 
fig1.add_trace(
    go.Bar(
        x=stage_means["Target"].astype(str),
        y=stage_means["Calcium"],
        name="Calcium (mg/dL)",
        marker_color=[STAGE_COLORS[s] for s in stage_means["Target"].astype(str)],
        opacity=0.9,
        text=stage_means["Calcium"].round(2),
        textposition="outside",
    ),
    
    row=1, col=1,
)
fig1.add_trace(
    go.Scatter(
        x=stage_means["Target"].astype(str),
        y=stage_means["Phosphorus"],
        name="Phosphorus (mg/dL)",
        mode="lines+markers",
        line=dict(color="#e74c3c", width=3, dash="dot"),
        marker=dict(size=10, color="#e74c3c", symbol="diamond"),
        yaxis="y2",
    ),
    row=1, col=1,
)
 
# Ca × P product (bone mineralisation risk; >55 is dangerous)
stage_means["CaP_Product"] = stage_means["Calcium"] * stage_means["Phosphorus"]
fig1.add_trace(
    go.Bar(
        x=stage_means["Target"].astype(str),
        y=stage_means["CaP_Product"],
        name="Ca×P Product",
        marker_color=[STAGE_COLORS[s] for s in stage_means["Target"].astype(str)],
        text=stage_means["CaP_Product"].round(1),
        textposition="outside",
        showlegend=False,
    ),
    row=1, col=2,
)
# Danger threshold line
fig1.add_hline(
    y=55, line_dash="dash", line_color="red", line_width=2,
    annotation_text="Risk threshold (55)", annotation_position="top left",
    row=1, col=2,
)
 
fig1.update_layout(
    title=dict(
        text="Calcium–Phosphorus Imbalance & Bone Health in CKD",
        font=dict(size=18),
    ),
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25, xanchor="center", x=0.25),
    yaxis=dict(title="Calcium (mg/dL)", showgrid=True, gridcolor="#ececec"),
    yaxis2=dict(
        title="Phosphorus (mg/dL)",
        overlaying="y", side="right",
        showgrid=False,
    ),
    yaxis3=dict(title="Ca × P Product", showgrid=True, gridcolor="#ececec"),
    height=500,
    font=dict(family="Arial", size=12),
    bargap=0.35,
)
fig1.update_xaxes(tickangle=-20)

The left panel 

Calcium stays almost perfectly flat — hovering around 9 mg/dL from Healthy Kidney all the way to Stage 5. This looks stable, but it's actually misleading — the body is working hard to keep it that way by pulling calcium out of bones.

Phosphorus rises sharply and consistently — from 2.5 mg/dL in healthy kidneys up to 7.0 mg/dL in Stage 5 (Kidney Failure). This is because healthy kidneys excrete excess phosphorus in urine, and as they fail, phosphorus accumulates in the blood.


Why calcium appears stable (the hidden mechanism)


When phosphorus rises, it binds with calcium in the blood, which would drop calcium. The parathyroid gland detects this and releases PTH (parathyroid hormone), which does three things to rescue blood calcium levels: pulls calcium from bones, reduces calcium lost in urine, and activates Vitamin D to absorb more from food. So your plot's flat calcium line represents a body constantly robbing its own skeleton to maintain balance.

### Q3 Which health measures (like Heart Rate or Bicarbonate) are linked to dangerously high Potassium levels that can cause heart problems?

In [24]:
HYPERK_THRESHOLD = 5.5          # mmol/L clinical cut-off
df["Hyperkalemia"] = (df["Potassium"] >= HYPERK_THRESHOLD).astype(int)
 
biomarkers = [
    "Heart_Rate", "Bicarbonate", "eGFR", "Sodium",
    "Serum_Creatinine", "Hemoglobin", "Chloride", "Phosphorus",
]
 
correlations = df[biomarkers + ["Potassium"]].corr()["Potassium"].drop("Potassium")
corr_df = correlations.reset_index()
corr_df.columns = ["Biomarker", "Correlation"]
corr_df = corr_df.sort_values("Correlation")
 
fig2 = go.Figure()
fig2.add_trace(
    go.Bar(
        x=corr_df["Correlation"],
        y=corr_df["Biomarker"],
        orientation="h",
        marker_color=[
            "#e74c3c" if c > 0 else "#3498db" for c in corr_df["Correlation"]
        ],
        text=[f"{c:+.3f}" for c in corr_df["Correlation"]],
        textposition="outside",
    )
)
fig2.add_vline(x=0, line_color="black", line_width=1.5)
fig2.update_layout(
    title=dict(
        text=(
            "Biomarkers Correlated with High Potassium (≥5.5 mmol/L)"
            "<br><sup>Positive = rises with Potassium | Negative = falls as Potassium rises</sup>"
        ),
        font=dict(size=16),
    ),
    xaxis=dict(title="Pearson Correlation with Potassium", zeroline=False),
    yaxis=dict(title=""),
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="white",
    height=420,
    font=dict(family="Arial", size=12),
    margin=dict(l=130),
)
 

### Q4) How is Bicarbonate (blood acidity) related to kidney function (eGFR)?

In [27]:
fig3 = px.scatter(
    df_sorted,
    x="eGFR",
    y="Bicarbonate",
    color="Target",
    color_discrete_map=STAGE_COLORS,
    category_orders={"Target": stage_order},
    opacity=0.55,
    labels={"eGFR": "eGFR (mL/min/1.73 m²)", "Bicarbonate": "Bicarbonate (mEq/L)"},
    title="Bicarbonate vs eGFR — Blood Acidity & Kidney Function",
)
 
# OLS trend line
m, b = np.polyfit(df["eGFR"], df["Bicarbonate"], 1)
x_range = np.linspace(df["eGFR"].min(), df["eGFR"].max(), 200)
fig3.add_trace(
    go.Scatter(
        x=x_range,
        y=m * x_range + b,
        mode="lines",
        name=f"Trend (r={df[['eGFR','Bicarbonate']].corr().iloc[0,1]:.2f})",
        line=dict(color="black", width=2.5, dash="dash"),
    )
)
# Clinical acidosis line
fig3.add_hline(
    y=22, line_dash="dot", line_color="red", line_width=2,
    annotation_text="Acidosis threshold (22 mEq/L)",
    annotation_position="bottom right",
)
fig3.update_layout(
    plot_bgcolor="#f9f9f9",
    paper_bgcolor="white",
    height=500,
    font=dict(family="Arial", size=12),
    legend=dict(title="CKD Stage"),
)
 
 